# Tutorial 99 — Live Demo: From Experimental Counts to Mutual Information

This notebook is intentionally more compact than the numbered feature tutorials, but it follows the same teaching pattern: first we explain what is happening, then the code shows the exact QCOM calls.

The demo uses the large plaintext count file in `example_data`, normalizes those counts into probabilities, prints the dominant bitstrings, and computes classical mutual information across a bipartition of the register.


## 1) Setup

Run this notebook from the `tutorials/` directory after installing the local package:

```bash
python -m pip install -e ".[dev,parquet,viz]"
```

The file path below is relative to `tutorials/`. QCOM's plaintext parser returns `(counts, total_count)`, where `counts` maps bitstrings to raw shot counts.


In [ ]:
from pathlib import Path

import qcom as qc

file_path = Path("../example_data/1_billion_3.0_4_rungs.txt")
print(file_path)


## 2) Load the count dictionary

The text reader expects lines of the form `<bitstring> <value>`. For experimental measurement data, the values are usually raw counts. We keep them as counts at first because count totals are meaningful: they tell us how many shots contributed to the empirical distribution.


In [ ]:
count_dict, total_counts = qc.parse_file(file_path, show_progress=False)

print("unique bitstrings:", len(count_dict))
print("total counts:", int(total_counts))
qc.print_most_probable_data(count_dict, 10)


## 3) Make the data type explicit

QCOM still accepts raw dictionaries for convenience, but typed containers remove ambiguity. `CountsData` validates that all keys are bitstrings of the same length and that the stored shot count matches the dictionary total.


In [ ]:
counts_data = qc.CountsData({key: int(value) for key, value in count_dict.items()}, source=str(file_path))
print(counts_data)
print("shots:", counts_data.shots)
print("sites:", counts_data.n_sites)


## 4) Normalize counts to probabilities

Entropy and mutual information are probability-distribution quantities. We therefore normalize the count dictionary before calling information metrics. The dictionary-returning path is shown first because it is lightweight and familiar; the container-returning path is useful when you want validated metadata to travel with the probabilities.


In [ ]:
prob_dict = qc.normalize_to_probabilities(counts_data)
probability_data = qc.normalize_to_probabilities(counts_data, return_data=True)

print("probability sum:", sum(prob_dict.values()))
print("container:", probability_data)
qc.print_most_probable_data(probability_data.to_dict(), 10)


## 5) Choose a bipartition

The bitstrings in this dataset have eight sites. Here we split the first four sites into region A and the last four sites into region B. QCOM uses a `configuration` list to mark regions: `0` for A and `1` for B in the classical mutual-information helpers.


In [ ]:
configuration = [0] * 4 + [1] * 4
print("configuration:", configuration)
print("bitstring length:", probability_data.n_sites)


## 6) Compute mutual information

The scalar default is convenient when you only need $I(A;B)$. Passing `return_components=True` returns `MutualInformationResult`, which also exposes the three entropies used internally:

$$
I(A;B) = H(A) + H(B) - H(AB).
$$


In [ ]:
mutual_information = qc.compute_mutual_information(
    prob_dict,
    configuration=configuration,
    base=2,
)
components = qc.compute_mutual_information(
    prob_dict,
    configuration=configuration,
    base=2,
    return_components=True,
)

print("I(A;B) bits:", mutual_information)
print("H(A):", components.h_a)
print("H(B):", components.h_b)
print("H(AB):", components.h_ab)
print("dataclass type:", type(components).__name__)


## 7) What this demo showed

In a few QCOM calls we moved from a raw experimental-style count file to a normalized probability distribution and then to an information-theoretic summary. The important habit is to keep counts and probabilities conceptually separate: counts carry shot statistics, while probabilities are the right input for entropy and mutual-information calculations.
